# Indexing ClueWeb09 Category B by OpenSearch for BM25 Model

- [clueweb09/catb](https://ir-datasets.com/clueweb09.html#clueweb09/catb)
- Prerequisite: corpus on disk via [dataset/clueweb09-catb](../../dataset/clueweb09-catb/README.md)

50.2M web pages. Text comes from `doc.default_text()` (HTML extraction,
title on the first line); the full extracted text is indexed with no
truncation, matching the Anserini/Indri literature convention. Extraction
failures are logged and skipped, never silently dropped.

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets opensearch-py dotenv

In [ ]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

### Index a Corpus for BM25 Model

In [ ]:
import ir_datasets
dataset_name = "clueweb09/catb"
dataset = ir_datasets.load(dataset_name)

In [ ]:
index_name = "clueweb09_catb_bm25"

In [ ]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0,
      # Disabled during bulk indexing; re-enabled after the run below.
      "refresh_interval": "-1"
    }
    # English corpus: rely on OpenSearch's default (standard) analyzer.
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "url": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

Document preparation

`default_text()` puts the extracted title on the first line; pages that
yield a single line (no newline) have no separate title. Extraction
failures land in `skipped` (log-and-skip).

In [ ]:
skipped = []

def to_action(doc):
    text = doc.default_text()
    nl = text.find("\n")
    title, body = (text[:nl], text[nl+1:]) if nl > 0 else ("", text)
    return {
        "_id": doc.doc_id,
        "_source": {
            "docid": doc.doc_id,
            "url": doc.url,
            "title": title,
            "text": body,
        }
    }

def prepare_documents(dataset):
    for doc in dataset.docs_iter():
        try:
            yield to_action(doc)
        except Exception as e:            # log-and-skip, report at the end
            skipped.append((doc.doc_id, str(e)[:200]))

### Benchmark a slice first

Measured on the first WARC file (35,582 docs, 2026-08): ~1,480 docs/s with
single-threaded extraction -> **~9.4 h** for the full 50.2M docs, ~4.1 KB/doc
of index (~206 GB total). Re-run this cell to recalibrate if the hardware
changes.

In [ ]:
import itertools, time

N = 10_000
t0 = time.time()
for _ in itertools.islice(prepare_documents(dataset), N):
    pass
rate = N / (time.time() - t0)
print(f"extraction rate: {rate:.0f} docs/s "
      f"-> full corpus lower bound {dataset.docs_count()/rate/3600:.1f} h (excl. indexing overhead)")

Indexing (expect roughly an overnight run)

In [ ]:
from opensearchpy.helpers import parallel_bulk

total = dataset.docs_count()   # 50,220,423

success, errors = 0, []
with tqdm(total=total, desc="Indexing") as bar:
    for ok, item in parallel_bulk(
        client,
        prepare_documents(dataset),
        index=index_name,
        chunk_size=500,
        thread_count=4,
        queue_size=4,
        request_timeout=600,
        raise_on_error=False,          # collect failures instead of aborting the run
        raise_on_exception=False,
    ):
        bar.update(1)                  # advances per actually-processed document
        success += ok
        if not ok:
            errors.append(item)

print(f"indexed: {success},  failed: {len(errors)},  extraction skipped: {len(skipped)}")
if errors:
    pprint.pprint(errors[:3])
if skipped:
    pprint.pprint(skipped[:10])

# Re-enable refresh now that bulk indexing is done, and make docs searchable.
client.indices.put_settings(index=index_name, body={"index": {"refresh_interval": "1s"}})
client.indices.refresh(index=index_name)
print("final count:", client.count(index=index_name)["count"])

---
### Completeness check

At 50M docs the scan-diff used by the smaller notebooks would hold ~3-5 GB
of ids in memory; the count comparison below is the default check. Fall back
to the scan-diff (see the msmarco notebooks) only if the counts disagree and
you need the exact missing ids -- and expect it to take a while.

In [ ]:
count = client.count(index=index_name)["count"]
expected = dataset.docs_count() - len(skipped)
print(f"index count: {count}  expected: {expected}  missing: {expected - count}")